# Day 8 — Week 2 Project: Cracking Caesar with Grover's Algorithm

**Theme:** Same problem as Week 1, new kind of computer

## Student Instructions
- **Only edit the code cells labeled `EDIT THIS CELL ONLY`.**
- Run the notebook from top to bottom.

## The Mission
We intercepted a secret message. It was encrypted with a Caesar cipher, and we don't know the key.
We do know one thing: **every message from this sender starts with "HI"**.

Knowing a small piece of the plain message is called a **known-plaintext attack**. At Bletchley Park in WW2,
codebreakers used this trick against Enigma. German weather reports always contained the word "WETTER".

We'll crack the message twice:
1. the **classical** way (Week 1 brute force)
2. the **quantum** way (Grover's algorithm)

Then we'll ask the honest question: how close are real quantum computers to doing this?

### If a cell runs too long
Some cells let you choose how many bits, qubits, or Grover rounds to use. **Big choices can take minutes, or even hours.**
Each of those cells says how long to expect, and the helper functions print an estimate before anything slow starts.
- A cell is still running while you see `[*]` next to it (Jupyter) or a spinning circle (VS Code).
- To stop it, press the **Interrupt** button (■) at the top of the notebook.
- If the notebook stops responding, **restart** it (Kernel → Restart), then re-run the Setup cell and the cells above where you were.

## Plan (about 75 minutes)
1. Week 1 recap: brute force *(10 min)*
2. Build a quantum oracle that tests keys *(20 min)*
3. Crack the message with Grover *(15 min)*
4. Classical vs. quantum: who wins, and by how much? *(10 min)*
5. Reality check: noise *(15 min)*
6. Wrap-up: what quantum computers do well, and what's still hard *(5 min)*

## 0. Setup

In [ ]:
# DO NOT EDIT
# If Qiskit is not installed, remove the # from the next line and run this cell again:
# %pip install qiskit qiskit-aer matplotlib

from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector
from qiskit_aer import Aer
from qiskit.visualization import plot_histogram

simulator = Aer.get_backend("aer_simulator")   # the same simulator as Day 5
try:
    from grover_viz import *
    print("Qiskit and the Grover helpers are ready!")
except ModuleNotFoundError as error:
    print("Could not find", error.name)
    print("Make sure grover_viz.py is in the same folder as this notebook.")
    print("(Google Colab: click the folder icon on the left and upload grover_viz.py.)")

## 1. Week 1 Recap: Caesar Cipher and Brute Force

One small change from Week 1: our alphabet has **32 symbols** instead of 26.
We add the space and `. , ! ? '`. Why 32? Because $32 = 2^5$, so every key fits **exactly** in **5 qubits**.

Remember the Week 1 **Math Moment**: shifting wraps around the alphabet with `% 26`.
With 32 symbols, we wrap around with `% 32` instead. The functions `caesar_encode` and `caesar_decode` work just like Week 1.

In [ ]:
# DO NOT EDIT
print("Alphabet:", ALPHABET32)
print("Number of symbols:", len(ALPHABET32))
for i, ch in enumerate(ALPHABET32):
    print(f"{ch!r}={i}", end="  ")

In [ ]:
# EDIT THIS CELL ONLY
my_message = "MEET ME AFTER SCHOOL!"
my_key = 7

secret = caesar_encode(my_message, my_key)
print("Encrypted:", secret)
print("Decrypted:", caesar_decode(secret, my_key))

### The intercepted message

In [ ]:
# DO NOT EDIT
intercepted = "UVHARNZJHAURH!BN ABZHYNOH.,R 'HNAH .. IHO?V TH' NPX'K"
known_letter = "H"            # every message from this sender starts with "HI"

print("Intercepted:", intercepted)
print(f"We know {known_letter!r} was encrypted into {intercepted[0]!r}")

### Classical brute force
Just like Week 1's `caesar_bruteforce`: try every key and **check** it.
In Week 1 *you* checked each try by reading it. Here the computer does the checking.

The check only uses information we already have. Does this key turn `"H"` into the first letter of the ciphertext?

In [ ]:
# DO NOT EDIT
def is_right_key(key):
    """The classical 'oracle': True if this key encrypts the known letter into the ciphertext's first letter."""
    return caesar_encode(known_letter, key) == intercepted[0]

classical_tries = 0
for key in range(32):
    classical_tries = classical_tries + 1
    if is_right_key(key):
        break

print(f"Classical search found the key after {classical_tries} tries.")
print("(We'll keep the message secret until the quantum computer cracks it too!)")

### Reflection
- What is the worst case for this search? The average?
- `is_right_key` checks **one** key per call. Keep that in mind for the quantum version.

## 2. Building a Quantum Oracle

On Day 6, the oracle was "cheating": we **told** it the key. A real codebreaker doesn't know the key!
The oracle must recognize the right key using only public information: `"H"` and the ciphertext.

So we build the quantum version of `is_right_key`, in three steps:

| Step | Classical version | Quantum version |
|---|---|---|
| 1. Encrypt | `caesar_encode("H", key)` | **add** H's number (7) to the 5 key qubits |
| 2. Check | `== intercepted[0]` | **flip the sign** if the qubits now spell the ciphertext letter |
| 3. Clean up | *(nothing)* | **subtract** 7 again, so the qubits hold the key guess again |

The big difference: the 5 qubits hold **all 32 keys at once**, so the oracle checks every key in one go.

### Step 1: Encryption is addition
A Caesar shift is just **adding numbers**. In Week 1 you built adders from logic gates (half adder and full adder).
Here is the quantum adder. It is built from controlled-NOT gates: a bit flips when every bit below it is 1, just like a carry.

With only 5 qubits, the adder wraps around **by itself**, like a car odometer rolling over from 99999 to 00000.
So we get the `% 32` for free!

In [ ]:
# DO NOT EDIT
plain_number = ALPHABET32.index(known_letter)
encrypt = add_constant(5, plain_number)
print(f"Encrypting {known_letter!r} means adding {plain_number}:")
display(encrypt.draw(fold=-1))

### Your Turn: test the adder like a truth table
Put **one** key into the qubits (no superposition), run the adder, and read the result.

- *Predict* which letter you'll get for `test_key = 3`, then run it.
- Try more keys. Does the adder agree with `caesar_encode`?

In [ ]:
# EDIT THIS CELL ONLY
test_key = 3

qc = QuantumCircuit(5)
qc.prepare_state(test_key)       # load the number test_key into the 5 qubits
qc.compose(encrypt, inplace=True)

result = Statevector(qc).probabilities_dict(decimals=6)   # no superposition, so only ONE possible result
bits = list(result)[0]
number = int(bits, 2)                           # binary -> number, like Day 1
print("Key", test_key, "-> qubits read", bits, "=", number, "=", ALPHABET32[number])
print("caesar_encode says:", caesar_encode(known_letter, test_key))

### Steps 2 and 3: check, then clean up
Here is the full oracle. Notice what it knows: `'H'` and the ciphertext letter. **It is never told the key.**

In [ ]:
# DO NOT EDIT
oracle = caesar_oracle(known_letter, intercepted[0])
display(oracle.draw(fold=-1))

### Does the oracle work?
Let's test the oracle on every key, **one at a time**, and see whose sign gets flipped.

Notice that this test is **classical brute force** again: one key per check.
Grover's algorithm will hand the oracle all 32 keys **at once**.

In [ ]:
# DO NOT EDIT
flipped = []
for key in range(32):
    qc = QuantumCircuit(5)
    qc.prepare_state(key)
    amplitude = Statevector(qc).evolve(oracle).data[key].real
    if amplitude < 0:
        flipped.append(key)
print("Keys the oracle flips:", flipped)

## 3. Crack It with Grover

Now give the oracle all 32 keys in superposition and let Grover's algorithm rotate the state toward the right key.
The bars are labeled with key numbers 0–31. The yellow bar is the key the oracle flips.
The picture only uses yellow to help *you* follow along; the algorithm never sees it.

> **Runtime:** This animation (4 rounds) takes about **25 seconds** to build.

In [ ]:
# DO NOT EDIT
animate_grover(5, oracle, labels=[str(k) for k in range(32)],
               title="Grover search over all 32 Caesar keys")

### Measure, like a real quantum computer would
On real hardware we can't watch the bars. We just measure.

### Try these ideas
- How few `shots` do you need to be confident? Try 1, 3, 10.

> **Runtime:** Runs in about a second, even with 10,000 shots.

In [ ]:
# EDIT THIS CELL ONLY
shots = 3

In [ ]:
# DO NOT EDIT
qc = grover_circuit(5, oracle)          # Hadamards, 4 rounds of (Caesar oracle + Diffuser), measure
job = simulator.run(transpile(qc, simulator), shots=shots)
counts = job.result().get_counts()
plot_counts(counts, labels=[str(k) for k in range(32)])

In [ ]:
# DO NOT EDIT
best_bits = max(counts, key=counts.get)   # the result we measured most often
quantum_key = int(best_bits, 2)
print("Measured key:", quantum_key)
print("Check it classically:", is_right_key(quantum_key))

In [ ]:
# DO NOT EDIT
print("Decrypted message:")
print(caesar_decode(intercepted, quantum_key))

## 4. Classical vs. Quantum: Who Wins, and by How Much?

Count the **oracle calls**. That's the expensive part in both versions.

In [ ]:
# DO NOT EDIT
grover_rounds = best_iterations(5)
print(f"Classical brute force (today's message): {classical_tries} checks")
print(f"Classical brute force (average):          {32 // 2} checks")
print(f"Grover's algorithm:                       {grover_rounds} oracle calls (+ 1 classical check of the answer)")

For 32 keys that's a small win. Real ciphers have **far** bigger key spaces.
The table assumes both computers can do **one billion** checks per second. That is very generous for a quantum computer!

In [ ]:
# DO NOT EDIT
from math import pi, sqrt

def how_long(checks, per_second=1e9):
    seconds = checks / per_second
    for unit, size in [("years", 3.156e7), ("days", 86400), ("hours", 3600), ("minutes", 60)]:
        if seconds >= size:
            value = seconds / size
            if value >= 1e9:
                return f"{value:.1e} {unit}"
            return f"{value:,.1f} {unit}" if value < 10 else f"{value:,.0f} {unit}"
    return f"{seconds:.1g} seconds"

ciphers = [("Our Caesar cipher", 5), ("a 20-bit key", 20), ("DES (1977)", 56),
           ("AES-128 (today)", 128), ("AES-256 (today)", 256)]

print(f"{'cipher':<20} {'key bits':>8} {'classical time':>26} {'Grover time':>26}")
for name, bits in ciphers:
    N = 2 ** bits
    classical = N / 2
    grover = pi / 4 * sqrt(N)
    print(f"{name:<20} {bits:>8} {how_long(classical):>26} {how_long(grover):>26}")

print("\n1.4e+10 means 1.4 x 10^10. For comparison, the universe is about 1.4e+10 years old.")

### Reflection
- Grover turns about $N/2$ checks into about $\frac{\pi}{4}\sqrt{N}$. With 128-bit keys, does that break AES?
- What simple change would make a cipher safe against Grover again? (Hint: what if the key were twice as long?)
- Grover's speed-up is called **quadratic**. A different quantum algorithm, **Shor's algorithm**, threatens the
  public-key encryption that protects websites (RSA) **much** more. That's why new "post-quantum" encryption
  standards were published in 2024.

## 5. Reality Check: Noise

Everything today ran on a **perfect** simulator. On Day 5 you saw that real qubits are **noisy**: every gate has a
small chance of an error.

First, how big is our code-breaking circuit once it is compiled for a real chip?
Real chips only have simple gates, and each qubit can only talk to its neighbors.

In [ ]:
# DO NOT EDIT
cost = circuit_cost(5, oracle)
for name, value in cost.items():
    print(f"{name:>24}: {value:,}")

Over two thousand CNOT gates! If each one fails even 1% of the time, almost every run hits at least one error.

Here is the same simple Grover search on a perfect simulator and on a noisy one, at different sizes
(takes a few seconds):

In [ ]:
# DO NOT EDIT
results = noise_experiment(qubit_counts=(2, 3, 4, 5), two_qubit_error=0.01)

### Try these ideas
Run the **Caesar crack** with noise.
- Start with `two_qubit_error = 0.01` (1% error per CNOT, close to many of today's chips).
- How small must the error be before the right key shows up again? Try `0.001` and `0.0001`.

> **Runtime:** Each run takes about a second, so try as many error rates as you like.

In [ ]:
# EDIT THIS CELL ONLY
two_qubit_error = 0.01
shots = 1000

counts = run_grover(5, oracle, shots=shots, two_qubit_error=two_qubit_error)
plot_counts(counts, marked=format(quantum_key, "05b"), labels=[str(k) for k in range(32)],
            title=f"Caesar crack with {100 * two_qubit_error:g}% error per CNOT")

### Reflection
- At what error rate does the key stand out clearly again?
- The fix researchers are working on is **quantum error correction**: spread one reliable "logical" qubit over many
  noisy physical qubits. Why does that make quantum computers need **many more** qubits?

## 6. Wrap-up: What Quantum Computers Do Well, and What's Still Hard

| Quantum computers are promising for... | ...but today's challenges are |
|---|---|
| **Searching** huge spaces: Grover, a quadratic speed-up (cipher keys, or guessing a password from its hash like in Week 1) | **Noise**: errors pile up in deep circuits (Section 5) |
| **Factoring**: Shor's algorithm, a threat to RSA | **Error correction** needs many physical qubits per logical qubit |
| **Simulating molecules and materials**: chemistry, batteries, medicine | **The oracle is expensive**: most of our circuit was the oracle, not Grover |
| Some **optimization and sampling** problems | **You can't peek**: you only get measurement samples, never the amplitudes |
| | **Not faster at everything**: email, games, and spreadsheets stay classical |

Compare this with the end of Week 1. What did classical computers do well that quantum computers *still* can't?

## Final Reflection Questions
1. Explain in your own words how the quantum oracle checks a key **without knowing the key**.
2. Why did the oracle need an "undo" step that the classical `is_right_key` didn't?
3. Grover needed 4 oracle calls and classical brute force needed 16 on average. Why is that gap so much bigger for a 128-bit key?
4. If you were designing encryption today, what would you do about Grover's algorithm?
5. What is the biggest thing stopping today's quantum computers from running this attack on a real cipher?

## Challenge: Crack a Classmate's Message
**Sender:** pick a secret key (0–31) and a message that starts with a letter your partner knows.
Give your partner **only** the ciphertext and that first letter.

In [ ]:
# EDIT THIS CELL ONLY  (sender)
my_message = "HELLO FROM THE QUANTUM SIDE!"
my_secret_key = 21

print("Send this ciphertext:", caesar_encode(my_message, my_secret_key))

**Codebreaker:** paste the ciphertext and the known first letter, then let Grover find the key.

In [ ]:
# EDIT THIS CELL ONLY  (codebreaker)
their_ciphertext = ",ZAADP GDBPI,ZPFJVCIJBPH!YZS"   # replace with your partner's ciphertext
their_first_letter = "H"

their_oracle = caesar_oracle(their_first_letter, their_ciphertext[0])
counts = run_grover(5, their_oracle, shots=5)
found_key = int(max(counts, key=counts.get), 2)
print("Grover found key:", found_key)
print("Message:", caesar_decode(their_ciphertext, found_key))